In [32]:
from sklearn.metrics import recall_score

exp2b_model = joblib.load("models/random_forest/random_forest_exp2b.joblib")
print("Researching probable class distribution changing\n")
print("Investigation 1: Class distribution over time\n")
temporal_distribution = random_forest_df.groupby("time_step")["is_illicit"].agg(
        total_transactions="count",
        illicit_transactions="sum"
    )

temporal_distribution["licit_transactions"] = (
    temporal_distribution["total_transactions"]
    - temporal_distribution["illicit_transactions"]
)

temporal_distribution["illicit_ratio"] = (
    temporal_distribution["illicit_transactions"]
    / temporal_distribution["total_transactions"]
)

print(temporal_distribution)

print("Investigation 2 and 3: Top feature drift and Statistical feature drift")
top_features = [
    "f54",
    "f6",
    "f15",
    "f56",
    "f91",
    "f164",
    "f42",
    "f66",
    "f60",
    "f133"
]

feature_means = (
    random_forest_df.groupby("time_step")[top_features].mean()
)
print(feature_means)

train_df = random_forest_df[random_forest_df["time_step"] <= 34]
validation_df = random_forest_df[
    (random_forest_df["time_step"] >= 35) &
    (random_forest_df["time_step"] <= 41)
]
test_df = random_forest_df[random_forest_df["time_step"] >= 42]
comparison = pd.DataFrame({
    "train_mean": train_df[feature_columns].mean(),
    "validation_mean": validation_df[feature_columns].mean(),
    "test_mean": test_df[feature_columns].mean()
})

comparison["validation_shift"] = (
    comparison["validation_mean"] - comparison["train_mean"]
)

comparison["test_shift"] = (
    comparison["test_mean"] - comparison["train_mean"]
)

comparison["difference"] = (
    comparison["test_shift"].abs()
)

comparison.sort_values(
    "difference",
    ascending=False
).head(20)

print("Investigation 6: Performance by individual time step")
results = []

for step in sorted(test_period_df["time_step"].unique()):

    mask = test_period_df["time_step"] == step

    X = test_period_df.loc[mask, feature_columns]
    y = test_period_df.loc[mask, "is_illicit"]

    pred = exp2b_model.predict(X)

    results.append({
        "time_step": step,
        "transactions": len(y),
        "recall": recall_score(y, pred),
    })

pd.DataFrame(results)

Researching probable class distribution changing

Investigation 1: Class distribution over time

           total_transactions  illicit_transactions  licit_transactions  \
time_step                                                                 
1                        2147                    17                2130   
2                        1117                    18                1099   
3                        1279                    11                1268   
4                        1440                    30                1410   
5                        1882                     8                1874   
6                         485                     5                 480   
7                        1203                   102                1101   
8                        1165                    67                1098   
9                         778                   248                 530   
10                        972                    18                 954   
11 

,time_step,transactions,recall
0,42,2154,0.803347
1,43,1370,0.000000
2,44,1591,0.041667
3,45,1221,0.000000
4,46,712,0.500000
5,47,846,0.000000
6,48,471,0.000000
7,49,476,0.017857


In [33]:
train_illicit = train_df[
    train_df["is_illicit"] == 1
]

test_illicit = test_df[
    test_df["is_illicit"] == 1
]


drift_illicit = pd.DataFrame({
    "train_mean": train_illicit[feature_columns].mean(),
    "test_mean": test_illicit[feature_columns].mean()
})


drift_illicit["difference"] = (
    drift_illicit["test_mean"]
    -
    drift_illicit["train_mean"]
).abs()


drift_illicit.sort_values(
    "difference",
    ascending=False
).head(20)

,train_mean,test_mean,difference
f102,-0.225939,1.325539,1.551478
f104,-0.221973,1.325598,1.547570
f101,-0.212792,1.313678,1.526469
f137,-0.194961,1.223733,1.418694
f140,-0.231479,1.144835,1.376314
f4,-0.055795,1.177754,1.233549
f138,-0.283904,0.943023,1.226928
f114,-0.061748,0.949853,1.011600
f152,-0.018733,0.937107,0.955840
f116,0.000002,0.945367,0.945365


In [34]:
from sklearn.metrics import recall_score
import pandas as pd

print("Investigation 7: Per-time-step probability analysis\n")
results = []

for step in sorted(test_period_df["time_step"].unique()):

    mask = test_period_df["time_step"] == step

    X = test_period_df.loc[mask, feature_columns]
    y = test_period_df.loc[mask, "is_illicit"]

    probs = exp2b_model.predict_proba(X)[:, 1]
    pred = (probs >= 0.5).astype(int)

    illicit_probs = probs[y == 1]
    licit_probs = probs[y == 0]

    results.append({
        "time_step": step,
        "transactions": len(y),
        "illicit": int(y.sum()),

        "mean_illicit_prob": illicit_probs.mean() if len(illicit_probs) else None,
        "median_illicit_prob": pd.Series(illicit_probs).median() if len(illicit_probs) else None,
        "min_illicit_prob": illicit_probs.min() if len(illicit_probs) else None,
        "max_illicit_prob": illicit_probs.max() if len(illicit_probs) else None,

        "mean_licit_prob": licit_probs.mean(),

        "recall": recall_score(y, pred)
    })

probability_results = pd.DataFrame(results)

probability_results

Investigation 7: Per-time-step probability analysis



,time_step,transactions,illicit,mean_illicit_prob,median_illicit_prob,min_illicit_prob,max_illicit_prob,mean_licit_prob,recall
0,42,2154,239,0.823396,0.999994,0.050795,0.999996,0.048872,0.803347
1,43,1370,24,0.208440,0.202979,0.078798,0.358515,0.066818,0.000000
2,44,1591,24,0.224782,0.160204,0.077242,0.967681,0.055932,0.041667
3,45,1221,5,0.147323,0.159198,0.070196,0.219268,0.058571,0.000000
4,46,712,2,0.476471,0.476471,0.246468,0.706474,0.040960,0.500000
5,47,846,22,0.191609,0.182193,0.088044,0.303391,0.053751,0.000000
6,48,471,36,0.201004,0.203520,0.037630,0.364097,0.051098,0.000000
7,49,476,56,0.212415,0.194163,0.083632,0.755600,0.040247,0.017857


In [35]:
print("Investigation 8: How many illicit transactions have high confidence?")

results = []

for step in sorted(test_period_df["time_step"].unique()):

    mask = test_period_df["time_step"] == step

    X = test_period_df.loc[mask, feature_columns]
    y = test_period_df.loc[mask, "is_illicit"]

    probs = exp2b_model.predict_proba(X)[:, 1]

    illicit_probs = probs[y == 1]

    results.append({
        "time_step": step,
        "illicit": len(illicit_probs),

        "P>=0.9": (illicit_probs >= 0.9).sum(),
        "P>=0.7": (illicit_probs >= 0.7).sum(),
        "P>=0.5": (illicit_probs >= 0.5).sum(),
        "P<0.5": (illicit_probs < 0.5).sum(),
        "mean_probability": illicit_probs.mean()
    })

confidence_results = pd.DataFrame(results)

confidence_results

Investigation 8: How many illicit transactions have high confidence?


,time_step,illicit,P>=0.9,P>=0.7,P>=0.5,P<0.5,mean_probability
0,42,239,181,188,192,47,0.823396
1,43,24,0,0,0,24,0.208440
2,44,24,1,1,1,23,0.224782
3,45,5,0,0,0,5,0.147323
4,46,2,0,1,1,1,0.476471
5,47,22,0,0,0,22,0.191609
6,48,36,0,0,0,36,0.201004
7,49,56,0,1,1,55,0.212415
